# Import

In [120]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [121]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Prep

## Loading variables

In [122]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [123]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [124]:
# Loading result graph and communities
with open(f"{RESULT_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{RESULT_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{RESULT_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

In [125]:
for c in communities:
    print(len(c))

2861
2629
2396
2311
2292
2102
1956
1585
1361
1352
548
409
43
22
20
15
14
14
11
7
5
4
3
3
3
2
2
2
2
2
2
1
1
1


## Helpful functions (big object, drop NAN)

In [126]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to NCBI

In [127]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi_dict = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi_dict.get, c)) for c in comms]
    return comms_ncbi

In [128]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))
with open(f"{RESULT_FOLDER}/result_communities_ncbi_selected.pkl", "wb") as f:
    pickle.dump(communities_ncbi, f)

[['6671', '51136', '91966', '8682', '9679', '23151', '3964', '30851', '50862', '64771', '54602', '10614', '23180', '4430', '51741', '7205', '1200', '55317', '10868', '219333', '8558', '7016', '23621', '9910', '29780', '253725', '214', '201161', '23034', '10788', '1808', '128338', '23111', '4134', '2055', '10807', '8835', '961', '5218', '2274', '8204', '1050', '138151', '115704', '23177', '22924', '90355', '5052', '57222', '9204', '4212', '10758', '9765', '23612', '9711', '6239', '9732', '29946', '8202', '23623', '9202', '116068', '9590', '58528', '80381', '80762', '10169', '6738', '55466', '4059', '81671', '55716', '26001', '3916', '9882', '64112', '29923', '23204', '79845', '9993', '26036', '10106', '51330', '80005', '2887', '64061', '3337', '57211', '11180', '23271', '51192', '310', '22848', '8019', '2119', '687', '5570', '10123', '27289', '5997', '6303', '152137', '23522', '780', '51232', '23071', '9098', '51119', '51255', '81566', '3985', '26268', '25959', '55332', '8763', '54468',

In [129]:
communities_ncbi_full = index_to_ncbi(communities,index_to_gene_distinct)
print(communities_ncbi_full)
print(len(communities_ncbi_full))
with open(f"{RESULT_FOLDER}/result_communities_ncbi.pkl", "wb") as f:
    pickle.dump(communities_ncbi_full, f)

[['18', '21', '29', '33', '34', '36', '41', '47', '50', '55', '125', '127', '130', '131', '135', '140', '147', '148', '151', '156', '157', '158', '164', '186', '190', '191', '197', '203', '204', '210', '211', '213', '214', '215', '216', '218', '229', '231', '239', '241', '242', '262', '290', '302', '310', '311', '316', '317', '325', '341', '344', '350', '359', '361', '362', '366', '369', '375', '383', '392', '395', '396', '397', '399', '400', '406', '429', '444', '467', '471', '476', '477', '478', '482', '490', '498', '501', '509', '513', '514', '522', '523', '525', '527', '537', '539', '570', '571', '575', '576', '580', '583', '586', '587', '593', '594', '610', '627', '636', '640', '643', '653', '660', '661', '668', '669', '683', '687', '688', '695', '715', '722', '725', '727', '728', '735', '752', '754', '765', '779', '780', '781', '784', '799', '825', '832', '835', '843', '844', '845', '859', '860', '873', '874', '875', '887', '896', '899', '908', '912', '916', '917', '923', '930', 

## NCBI to HGNC

In [130]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        symbols = [ncbi_to_hgnc_dict.get(n) for n in community]
        comms_HGNC.append(symbols)
    return comms_HGNC

In [131]:
# # NCBI to HGNC symbol
# def ncbi_to_HGNC(comms_ncbi):
#     comms_HGNC = []
#     for community in comms_ncbi:
#         mg = mygene.MyGeneInfo()
#         entrez_ids = [str(e) for e in community]

#         results = mg.querymany(
#             entrez_ids,
#             scopes="entrezgene",
#             fields="symbol",
#             species="human"
#         )

#         # Build a mapping: input ID -> symbol (or None)
#         id_to_symbol = {}
#         for r in results:
#             q = str(r.get("query"))
#             id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

#         # Preserve original order
#         symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
#         comms_HGNC.append(symbols)
#     return comms_HGNC


In [132]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)
COMMUNITIES_HGNC_full = ncbi_to_HGNC(communities_ncbi_full)

In [133]:
print(len(COMMUNITIES_HGNC))

13


In [134]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)
COMMUNITIES_HGNC_full = drop_nan_from_communities(COMMUNITIES_HGNC_full)

Community 0: dropped 0 NaN entries
Community 1: dropped 4 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 4 NaN entries
Community 5: dropped 0 NaN entries
Community 6: dropped 0 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 3 NaN entries
Community 10: dropped 1 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 0 NaN entries

Total dropped across all communities: 12
Community 0: dropped 0 NaN entries
Community 1: dropped 7 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 19 NaN entries
Community 5: dropped 0 NaN entries
Community 6: dropped 0 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 11 NaN entries
Community 10: dropped 1 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 0 NaN entries
Community 13: dropped 0 NaN entries
Comm

In [135]:
with open(f"{RESULT_FOLDER}/result_communities_HGNC_selected.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC, f)
with open(f"{RESULT_FOLDER}/result_communities_HGNC.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC_full, f)

In [136]:
print(len(COMMUNITIES_HGNC))
print(len(COMMUNITIES_HGNC_full))

13
34


# Categoization Prep

### GO-slim

In [137]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [138]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [139]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [140]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [141]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [142]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [143]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=300)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [144]:
# Specific for Reactome: build level map first
reactome_level1 = build_reactome_level_map(level = 1)

https://reactome.org/ContentService/data/eventsHierarchy/9606


# Run Enrichment Analysis

In [145]:
TERM_SCORE_CAP = 0.001
PERCENTAGE = 0.1

In [146]:
def enrichment(communities,
               term_score_cap,
               percentage, 
               db,
               term_to_category):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=db,
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["Category"] = filtered["Term"].apply(lambda term: term_to_category(term))

        # Get empty count
        empty_count = (filtered["Category"].apply(len) == 0).sum()
        
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "genes_involved": involved,
            "n_involved": len(involved),
            "n_not_involved": len(not_involved)
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

### GO

In [147]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["id"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["id"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "n_involved": len(involved),
            "n_not_involved": len(not_involved),
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

In [148]:
term = "Nuclear Pore Organization (GO:0006999)"
print(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))

{'GO:0009987'}


In [149]:
go_important_terms, go_community_coverage = enrichment(COMMUNITIES_HGNC,
                                                       TERM_SCORE_CAP,
                                                       PERCENTAGE,
                                                       ['GO_Biological_Process_2023',
                                                        'GO_Molecular_Function_2023',
                                                        'GO_Cellular_Component_2023'],
                                                       lambda term: [go[id].name for id in list(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))])

Size of community: 1097
Number of filtered terms: 68
Number of unmapped terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_58588\3315107404.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
38,0,Negative Regulation Of Cilium Assembly (GO:1902018),7/14,3.112898e-04,[biological regulation]
46,0,Reticulophagy (GO:0061709),7/16,7.872433e-04,[cellular process]
36,0,Protein K48-linked Deubiquitination (GO:0071108),9/24,2.484139e-04,[cellular process]
3997,0,Cul3-RING Ubiquitin Ligase Complex (GO:0031463),12/35,9.522100e-06,[protein-containing complex]
39,0,Regulation Of Cell Morphogenesis (GO:0022604),10/31,3.112898e-04,[biological regulation]
10,0,Regulation Of TORC1 Signaling (GO:1903432),16/50,1.573329e-06,[biological regulation]
3431,0,Phosphatidylinositol-3-Phosphate Binding (GO:0032266),12/40,4.119084e-05,[binding]
49,0,Negative Regulation Of BMP Signaling Pathway (GO:0030514),11/43,9.976708e-04,[biological regulation]
3424,0,Cysteine-Type Deubiquitinase Activity (GO:0004843),25/98,6.355289e-09,[catalytic activity]
24,0,Endocytic Recycling (GO:0032456),16/64,3.321460e-05,"[cellular process, localization]"


Size of community: 1140
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
1072,1,RNA Polymerase II Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000978),114/1122,1.201728e-07,[binding]
1071,1,RNA Polymerase II Transcription Regulatory Region Sequence-Specific DNA Binding (GO:0000977),124/1225,5.169911e-08,[binding]
1073,1,Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000987),111/1098,1.811394e-07,[binding]


...

Size of community: 655
Number of filtered terms: 1
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
1163,7,cullin-RING Ubiquitin Ligase Complex (GO:0031461),22/174,0.000011,[protein-containing complex]


Size of community: 371
Number of filtered terms: 5
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
17,11,Olfactory Receptor Activity (GO:0004984),306/362,0.000000e+00,[molecular transducer activity]
2,11,Detection Of Chemical Stimulus Involved In Sensory Perception Of Smell (GO:0050911),112/139,1.855031e-173,[response to stimulus]
1,11,Detection Of Chemical Stimulus Involved In Sensory Perception (GO:0050907),113/141,2.232478e-174,[response to stimulus]
0,11,Sensory Perception Of Smell (GO:0007608),183/230,9.537622e-291,[multicellular organismal process]
3,11,Sensory Perception Of Chemical Stimulus (GO:0007606),82/110,2.211176e-120,[multicellular organismal process]


8 out of 13 communities had significant GO terms.


In [150]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1097,Negative Regulation Of Cilium Assembly (GO:190...,7/14,3.112898e-04,[biological regulation],GO_Biological_Process_2023,3.577893e-06,0.0,0.0,17.335780,2.174034e+02,TCHP;LIMK2;TBC1D30;TESK1;CDK10;EVI5L;MAP4,0.500000
1,0,1097,Reticulophagy (GO:0061709),7/16,7.872433e-04,[cellular process],GO_Biological_Process_2023,1.082201e-05,0.0,0.0,13.481957,1.541517e+02,UFM1;RETREG3;DDRGK1;UBA5;UFC1;SNX33;RETREG1,0.437500
2,0,1097,Protein K48-linked Deubiquitination (GO:0071108),9/24,2.484139e-04,[cellular process],GO_Biological_Process_2023,2.688305e-06,0.0,0.0,10.416176,1.336041e+02,OTUB2;OTUD4;OTUD5;USP25;VCPIP1;USP20;USP5;OTUD...,0.375000
3,0,1097,Cul3-RING Ubiquitin Ligase Complex (GO:0031463),12/35,9.522100e-06,[protein-containing complex],GO_Cellular_Component_2023,1.798205e-07,0.0,0.0,9.078742,1.410047e+02,KLHL9;KLHL25;KCTD10;KLHL8;KLHL21;KCTD13;SPOPL;...,0.342857
4,0,1097,Regulation Of Cell Morphogenesis (GO:0022604),10/31,3.112898e-04,[biological regulation],GO_Biological_Process_2023,3.641881e-06,0.0,0.0,8.271783,1.035876e+02,ZRANB1;CLDN4;FMNL2;ZMYM4;ZMYM6;FITM2;PHIP;NOL3...,0.322581
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1231,11,371,Olfactory Receptor Activity (GO:0004984),306/362,0.000000e+00,[molecular transducer activity],GO_Molecular_Function_2023,0.000000e+00,0.0,0.0,1645.422527,inf,OR7G1;OR8I2;OR9K2;OR2M7;OR2M5;OR52N1;OR2M4;OR2...,0.845304
1232,11,371,Detection Of Chemical Stimulus Involved In Sen...,112/139,1.855031e-173,[response to stimulus],GO_Biological_Process_2023,3.273585e-174,0.0,0.0,313.945946,1.254101e+05,OR2M7;OR2M5;OR2M4;OR2M3;OR2M2;OR2T12;OR2T10;OR...,0.805755
1233,11,371,Detection Of Chemical Stimulus Involved In Sen...,113/141,2.232478e-174,[response to stimulus],GO_Biological_Process_2023,2.626445e-175,0.0,0.0,306.604790,1.232511e+05,OR2M7;OR2M5;OR2M4;OR2M3;OR2M2;OR2T12;OR2T10;OR...,0.801418
1234,11,371,Sensory Perception Of Smell (GO:0007608),183/230,9.537622e-291,[multicellular organismal process],GO_Biological_Process_2023,5.610366e-292,0.0,0.0,405.557492,2.719791e+05,OR8I2;OR1C1;OR2M7;OR2M5;OR2M4;OR2M3;OR2M2;OR2T...,0.795652


In [151]:
go_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1097,"[ABCB6, ABRAXAS2, ACE, ACP3, ACTL6B, ADGRA2, A...",740,357
1,1,1140,"[ATF7, CC2D1A, CCDC106, CIC, DLX6, DMTF1, FOXP...",126,1014
2,2,1021,"[AIRE, ALX3, ARTN, ASCL4, ASTN1, ATOH1, ATP8A2...",248,773
3,3,1033,"[ABL2, ABLIM1, ACKR3, ACTB, ACTN1, ACTN4, ACTR...",1013,20
4,4,828,[],0,828
5,5,848,"[CAMLG, CCDC47, EMC1, EMC2, EMC3, EMC6, EMC7, ...",14,834
6,6,848,"[ADGRE5, AEBP1, AIF1L, ANGPTL2, ANXA5, APLP2, ...",141,707
7,7,655,"[ANKRD9, APPBP2, DCAF1, DCAF11, DCAF4, DCAF5, ...",22,633
8,8,654,[],0,654
9,9,643,[],0,643


### KEGG

In [152]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [153]:
kegg_important_terms, kegg_community_coverage = enrichment(communities = COMMUNITIES_HGNC,
                                 term_score_cap = TERM_SCORE_CAP,
                                 percentage = PERCENTAGE,
                                 db = ['KEGG_2021_Human'],
                                 term_to_category = lambda term: get_kegg_level2(name_to_id.get(term.lower())))

Size of community: 1140
Number of filtered terms: 1
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_58588\3315107404.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,1,Herpes simplex virus 1 infection,79/498,4.141628e-15,[Infectious disease: viral]


Size of community: 1021
Number of filtered terms: 1
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Neuroactive ligand-receptor interaction,39/341,0.000288,[Signaling molecules and interaction]


Size of community: 1033
Number of filtered terms: 80
Number of unmapped terms: 3


,Community Index,Term,Overlap,Adjusted P-value,Category
3,3,Adherens junction,31/71,8.568575e-20,[Cellular community - eukaryotes]
11,3,TGF-beta signaling pathway,28/94,3.193299e-13,[Signal transduction]
63,3,Circadian rhythm,9/31,6.808603e-05,[Environmental adaptation]
25,3,p53 signaling pathway,20/73,4.433284e-09,[Cell growth and death]
57,3,Bladder cancer,11/41,2.068727e-05,[Cancer: specific types]
38,3,Viral myocarditis,16/60,2.455129e-07,[Cardiovascular disease]
0,3,MAPK signaling pathway,78/294,6.158280e-32,[Signal transduction]
20,3,Small cell lung cancer,24/92,3.179810e-10,[Cancer: specific types]
15,3,NF-kappa B signaling pathway,27/104,2.741954e-11,[Signal transduction]
13,3,TNF signaling pathway,29/112,5.100388e-12,[Signal transduction]


Size of community: 371
Number of filtered terms: 1
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Olfactory transduction,362/440,0.0,[Sensory system]


4 out of 13 communities had significant GO terms.


In [154]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,1,1140,Herpes simplex virus 1 infection,79/498,4.141628e-15,[Infectious disease: viral],KEGG_2021_Human,9.003539e-17,0.0,0.0,3.277043,121.074712,ZNF177;ZNF133;ZNF254;ZNF573;ZNF253;ZNF175;ZNF2...,0.158635
1,2,1021,Neuroactive ligand-receptor interaction,39/341,2.876515e-04,[Signaling molecules and interaction],KEGG_2021_Human,2.195813e-06,0.0,0.0,2.456141,32.000958,ADCYAP1R1;CALCB;SCT;GIPR;CSH1;PMCH;GRIK1;GPR83...,0.114370
2,3,1033,Adherens junction,31/71,8.568575e-20,[Cellular community - eukaryotes],KEGG_2021_Human,1.428096e-21,0.0,0.0,14.639147,702.648959,CTNND1;LEF1;PTPRM;PTPRJ;IQGAP1;ACTB;PTPRF;IGF1...,0.436620
3,3,1033,TGF-beta signaling pathway,28/94,3.193299e-13,[Signal transduction],KEGG_2021_Human,1.596650e-14,0.0,0.0,7.978712,253.470003,BMPR2;ACVR1B;PPP2CA;PITX2;SKP1;ACVR1;SMAD1;CDK...,0.297872
4,3,1033,Circadian rhythm,9/31,6.808603e-05,[Environmental adaptation],KEGG_2021_Human,1.815627e-05,0.0,0.0,7.568581,82.622375,PER1;FBXW11;BHLHE40;PRKAG2;RORA;NR1D1;BTRC;CLO...,0.290323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,3,1033,Kaposi sarcoma-associated herpesvirus infection,24/193,2.150561e-04,[Infectious disease: viral],KEGG_2021_Human,6.093258e-05,0.0,0.0,2.645727,25.678745,RB1;LYN;CREBBP;LEF1;EIF2AK2;HLA-A;FOS;HLA-G;HI...,0.124352
79,3,1033,JAK-STAT signaling pathway,20/162,8.130890e-04,[Signal transduction],KEGG_2021_Human,2.676418e-04,0.0,0.0,2.617382,21.530224,PDGFRB;PDGFRA;IL11;CSF3;CREBBP;TSLP;IL15;FHL1;...,0.123457
80,3,1033,NOD-like receptor signaling pathway,22/181,5.506521e-04,[Immune system],KEGG_2021_Human,1.697844e-04,0.0,0.0,2.574050,22.345282,YWHAE;HSP90AB1;ERBIN;CYBB;CYBA;TANK;MAPK13;OAS...,0.121547
81,3,1033,Salmonella infection,30/249,5.604342e-05,[Infectious disease: bacterial],KEGG_2021_Human,1.447788e-05,0.0,0.0,2.560538,28.531784,CYFIP2;HSP90AB1;ROCK2;LEF1;ACTB;HSP90B1;RRAS;C...,0.120482


In [155]:
kegg_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1097,[],0,1097
1,1,1140,"[RBAK, ZFP1, ZFP14, ZFP37, ZFP82, ZFP90, ZNF10...",79,1061
2,2,1021,"[ADCYAP1R1, CALCB, CRHR2, CSH1, GABRA3, GABRA4...",39,982
3,3,1033,"[ABL2, ABLIM1, ACKR3, ACTB, ACTN1, ACTN4, ACTR...",552,481
4,4,828,[],0,828
5,5,848,[],0,848
6,6,848,[],0,848
7,7,655,[],0,655
8,8,654,[],0,654
9,9,643,[],0,643


### Reactome

In [156]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [157]:
reactome_important_terms, reactome_community_coverage = enrichment(COMMUNITIES_HGNC,
                                      TERM_SCORE_CAP,
                                      PERCENTAGE,
                                      ['Reactome_2022'],
                                      lambda term: reactome_level1.get(term.split(" ")[-1],[]))

Size of community: 1097
Number of filtered terms: 9
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_58588\3315107404.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
7,0,RHOG GTPase Cycle R-HSA-9013408,17/74,4.419019e-05,[Signal Transduction]
0,0,Antigen Processing: Ubiquitination And Proteasome Degradation R-HSA-983168,50/307,3.828190e-09,[Immune System]
10,0,CDC42 GTPase Cycle R-HSA-9013148,23/149,5.494989e-04,[Signal Transduction]
9,0,RAC1 GTPase Cycle R-HSA-9013149,27/178,1.391760e-04,[Signal Transduction]
6,0,Neddylation R-HSA-8951664,34/237,3.567037e-05,[Metabolism of proteins]
2,0,Class I MHC Mediated Antigen Processing And Presentation R-HSA-983169,52/378,2.782526e-07,[Immune System]
1,0,RHO GTPase Cycle R-HSA-9012999,60/441,3.293017e-08,[Signal Transduction]
4,0,"Signaling By Rho GTPases, Miro GTPases And RHOBTB3 R-HSA-9716542",70/660,1.684564e-05,[Signal Transduction]
5,0,Signaling By Rho GTPases R-HSA-194315,68/644,2.543030e-05,[Signal Transduction]


Size of community: 1021
Number of filtered terms: 1
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Cell-cell Junction Organization R-HSA-421270,14/61,0.000798,[Cell-Cell communication]


...

Size of community: 848
Number of filtered terms: 5
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
2,6,Assembly Of Collagen Fibrils And Other Multimeric Structures R-HSA-2022090,13/57,1.234336e-04,[Extracellular matrix organization]
1,6,Collagen Formation R-HSA-1474290,17/90,6.563983e-05,[Extracellular matrix organization]
3,6,CDC42 GTPase Cycle R-HSA-9013148,21/149,2.173238e-04,[Signal Transduction]
0,6,RHO GTPase Cycle R-HSA-9012999,49/441,5.051981e-07,[Signal Transduction]
4,6,Extracellular Matrix Organization R-HSA-1474244,31/291,3.276598e-04,[Extracellular matrix organization]


Size of community: 371
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
2,11,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,354/393,0.0,[Sensory Perception]
1,11,Olfactory Signaling Pathway R-HSA-381753,354/401,0.0,[Sensory Perception]
0,11,Sensory Perception R-HSA-9709957,354/616,0.0,[Sensory Perception]


5 out of 13 communities had significant GO terms.


In [158]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1097,RHOG GTPase Cycle R-HSA-9013408,17/74,4.419019e-05,[Signal Transduction],Reactome_2022,3.834290e-07,0.0,0.0,5.204386,76.890178,DOCK5;ANKLE2;DOCK4;MCAM;ITSN1;ARHGEF16;ARHGAP1...,0.229730
1,0,1097,Antigen Processing: Ubiquitination And Proteas...,50/307,3.828190e-09,[Immune System],Reactome_2022,4.152050e-12,0.0,0.0,3.464782,90.802986,LRSAM1;UBE2D4;UBE3C;RNF19B;UBE2Z;TRIM9;NPEPPS;...,0.162866
2,0,1097,CDC42 GTPase Cycle R-HSA-9013148,23/149,5.494989e-04,[Signal Transduction],Reactome_2022,6.555843e-06,0.0,0.0,3.191385,38.089672,FARP1;RASGRF2;ITSN1;ARHGEF16;ARHGAP1;MYO9B;CDC...,0.154362
3,0,1097,RAC1 GTPase Cycle R-HSA-9013149,27/178,1.391760e-04,[Signal Transduction],Reactome_2022,1.509501e-06,0.0,0.0,3.133651,42.002616,DOCK5;DOCK4;RASGRF2;ARHGAP1;ARHGAP15;IQGAP2;AR...,0.151685
4,0,1097,Neddylation R-HSA-8951664,34/237,3.567037e-05,[Metabolism of proteins],Reactome_2022,2.708163e-07,0.0,0.0,2.946397,44.554896,CUL9;CUL7;DCUN1D3;KLHL13;DDA1;SOCS2;SOCS3;NUB1...,0.143460
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180,6,848,RHO GTPase Cycle R-HSA-9012999,49/441,5.051981e-07,[Signal Transduction],Reactome_2022,7.724741e-10,0.0,0.0,2.934919,61.578769,PLXND1;WIPF1;NCF2;DOCK9;SOWAHC;FAM13A;MSI2;ARH...,0.111111
181,6,848,Extracellular Matrix Organization R-HSA-1474244,31/291,3.276598e-04,[Extracellular matrix organization],Reactome_2022,2.505045e-06,0.0,0.0,2.757047,35.558202,FBN2;COL18A1;COL15A1;LAMA2;PCOLCE2;LAMA4;COL12...,0.106529
182,11,371,Expression And Translocation Of Olfactory Rece...,354/393,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,10459.819005,inf,OR7G1;OR8I2;OR9K2;OR11H2;OR11H1;OR11H4;OR2M7;O...,0.900763
183,11,371,Olfactory Signaling Pathway R-HSA-381753,354/401,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,8675.879850,inf,OR7G1;OR8I2;OR9K2;OR11H2;OR11H1;OR11H4;OR2M7;O...,0.882793


In [159]:
reactome_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1097,"[ABCD3, ABR, AKAP12, ANKFY1, ANKLE2, AREL1, AR...",135,962
1,1,1140,[],0,1140
2,2,1021,"[CADM2, CADM3, CDH10, CDH19, CDH4, CDH8, CLDN1...",14,1007
3,3,1033,"[ABL2, ABLIM1, ACKR3, ACTN1, ACTN4, ACTR2, ACT...",682,351
4,4,828,[],0,828
5,5,848,[],0,848
6,6,848,"[ADD3, AMIGO2, ARAP2, ARHGAP26, ARHGAP27, ARHG...",80,768
7,7,655,[],0,655
8,8,654,[],0,654
9,9,643,[],0,643


### Disease Data Sets

In [160]:
# disease_term_score_cap = 0.001
# disease_percentage = 0.1
# important_diseases = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value"])

In [161]:
# # Disease-gene enrichment libraries
# disease_sets = [
#     'DisGeNET_2020', # curated gene–disease associations
#     'GWAS_Catalog_2023', # genome-wide association hits
#     'OMIM_Disease', # Mendelian disorders
#     'Jensen_DISEASES' # text-mined associations
# ]

# # # Disease-gene enrichment Analysis; save terms with small size and high p-value
# i = 0
# for community in communities_HGNC:
#     # Gene Ontology enrichment
#     enr_disease = gp.enrichr(
#         gene_list=community,
#         gene_sets=disease_sets,
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     enr_disease_df = enr_disease.results.sort_values('Adjusted P-value')
#     print(f"Size of community: {len(community)}")

#     mask =  (enr_disease_df["Adjusted P-value"] < disease_term_score_cap) & (enr_disease_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > disease_percentage))
        
#     filtered = enr_disease_df[mask].copy()
#     if not filtered.empty:
#         filtered.loc[:, "Community Index"] = i
#         filtered.loc[:, "Community Size"] = len(community)
#         important_diseases = pd.concat([important_diseases, filtered], ignore_index=True)

#     display(HTML(filtered[['Term','Overlap','Adjusted P-value']].head(10).to_html(max_cols=None)))
#     i += 1

# Important Terms df

In [162]:
community_coverage_combined = go_community_coverage.copy()

community_coverage_combined["genes_involved"] = [
    set(a) | set(b) | set(c)
    for a, b, c in zip(go_community_coverage["genes_involved"], kegg_community_coverage["genes_involved"], reactome_community_coverage["genes_involved"])
]
community_coverage_combined["n_involved"] = community_coverage_combined["genes_involved"].apply(len)
community_coverage_combined["n_not_involved"] = community_coverage_combined["n_genes"] - community_coverage_combined["n_involved"]
community_coverage_combined["percentage_involved"] = community_coverage_combined["n_involved"] / community_coverage_combined["n_genes"]

In [163]:
community_coverage_combined

,community,n_genes,genes_involved,n_involved,n_not_involved,percentage_involved
0,0,1097,"{PARD6B, CST3, KLHL9, MAP4, SLF2, MEIS2, FAM53...",752,345,0.685506
1,1,1140,"{FOXP4, ZNF518B, ZNF468, SCX, ZNF275, ZSCAN16,...",148,992,0.129825
2,2,1021,"{OLIG3, ZIC4, MIXL1, CDH19, HPCAL4, DMRT2, LBX...",278,743,0.272282
3,3,1033,"{BIRC2, VIM, PPP1R16B, NGFR, TAOK1, SMG1, ETS1...",1025,8,0.992256
4,4,828,{},0,828,0.000000
5,5,848,"{MMGT1, EMC3, UBL4A, EMC9, TRAM2, CCDC47, EMC8...",14,834,0.016509
6,6,848,"{SNTB2, PCDH7, PDIA5, EFS, GOLM1, TNC, ARHGEF4...",185,663,0.218160
7,7,655,"{KCTD5, KCTD2, KLHDC10, APPBP2, ANKRD9, PCMTD1...",22,633,0.033588
8,8,654,{},0,654,0.000000
9,9,643,{},0,643,0.000000


In [164]:
comm_to_involved_pct = dict(zip(community_coverage_combined["community"], community_coverage_combined["percentage_involved"]))

with open(DISEASE_FOLDER + "comm_to_involved_pct.json", "w") as f:
    json.dump(comm_to_involved_pct, f, indent=2)


In [165]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1097,Negative Regulation Of Cilium Assembly (GO:190...,7/14,3.112898e-04,[biological regulation],GO_Biological_Process_2023,3.577893e-06,0.0,0.0,17.335780,2.174034e+02,TCHP;LIMK2;TBC1D30;TESK1;CDK10;EVI5L;MAP4,0.500000
56,0,1097,Supramolecular Fiber Organization (GO:0097435),38/316,3.962804e-04,[cellular process],GO_Biological_Process_2023,4.752119e-06,0.0,0.0,2.404026,2.946595e+01,TNFAIP6;AVIL;PLOD3;ARHGAP12;CAMSAP2;RND1;CST3;...,0.120253
55,0,1097,Regulation Of Intracellular Signal Transductio...,36/297,5.707200e-04,[biological regulation],GO_Biological_Process_2023,7.010893e-06,0.0,0.0,2.423478,2.876195e+01,PHLPP1;DYRK2;RNF34;ZFAND6;RASGRF2;ITSN1;ARHGAP...,0.121212
54,0,1097,Negative Regulation Of Cell Population Prolife...,47/379,2.533740e-05,[biological regulation],GO_Biological_Process_2023,1.482153e-07,0.0,0.0,2.503835,3.937181e+01,KANK2;BTG3;CEBPA;NDRG4;CTBP1;MAGED1;CTCF;PPM1D...,0.124011
53,0,1097,Vesicle-Mediated Transport (GO:0016192),52/411,3.880336e-06,"[cellular process, localization]",GO_Biological_Process_2023,1.658597e-08,0.0,0.0,2.570372,4.604747e+01,WASHC4;DENND1B;ITSN1;ARHGAP1;WASHC2C;FMN2;RAB2...,0.126521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1233,11,371,Detection Of Chemical Stimulus Involved In Sen...,113/141,2.232478e-174,[response to stimulus],GO_Biological_Process_2023,2.626445e-175,0.0,0.0,306.604790,1.232511e+05,OR2M7;OR2M5;OR2M4;OR2M3;OR2M2;OR2T12;OR2T10;OR...,0.801418
1232,11,371,Detection Of Chemical Stimulus Involved In Sen...,112/139,1.855031e-173,[response to stimulus],GO_Biological_Process_2023,3.273585e-174,0.0,0.0,313.945946,1.254101e+05,OR2M7;OR2M5;OR2M4;OR2M3;OR2M2;OR2T12;OR2T10;OR...,0.805755
1231,11,371,Olfactory Receptor Activity (GO:0004984),306/362,0.000000e+00,[molecular transducer activity],GO_Molecular_Function_2023,0.000000e+00,0.0,0.0,1645.422527,inf,OR7G1;OR8I2;OR9K2;OR2M7;OR2M5;OR52N1;OR2M4;OR2...,0.845304
1502,11,371,Olfactory Signaling Pathway R-HSA-381753,354/401,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,8675.879850,inf,OR7G1;OR8I2;OR9K2;OR11H2;OR11H1;OR11H4;OR2M7;O...,0.882793


In [166]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [167]:
# def run_enrichment_func(community,term_score_cap,percentage):
#     # GO df
#     enr_go = gp.enrichr(
#         gene_list=community,
#         gene_sets=['GO_Biological_Process_2023',
#                 'GO_Molecular_Function_2023',
#                 'GO_Cellular_Component_2023'],
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     GO_df = enr_go.results
#     mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     GO_df = GO_df[mask].copy()   
    
#     # KEGG df
#     enr_kegg = gp.enrichr(
#         gene_list=community,
#         gene_sets=['KEGG_2021_Human'],
#         organism='Human',
#         outdir=None
#     )
#     KEGG_df = enr_kegg.results
#     mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     KEGG_df = KEGG_df[mask].copy() 
       
#     # Reactome df
#     enr_reactome = gp.enrichr(
#         gene_list=community,
#         gene_sets=['Reactome_2022'],
#         organism='Human',
#         outdir=None
#     )
#     Reactome_df = enr_reactome.results  
#     mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     Reactome_df = Reactome_df[mask].copy()
    
    
#     all_df = [GO_df,KEGG_df,Reactome_df]
#     # build result df by concatenating
#     result = pd.concat(all_df, ignore_index=True)
#     return result

In [168]:
# from json import JSONDecodeError

# # ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
# _ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

# def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
#     """
#     Calls user's run_enrichment_func(community) with retries + memoization.
#     Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
#     """
#     # Ensure we always pass a list of gene symbols (never a bare string)
#     genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
#     if len(genes) == 0:
#         return pd.DataFrame()

#     key = tuple(sorted(genes))
#     if key in _ENR_CACHE:
#         return _ENR_CACHE[key].copy()

#     for a in range(retries):
#         try:
#             df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
#             if df is None:
#                 # treat as transient failure to trigger retry
#                 raise RuntimeError("run_enrichment_func returned None")
#             _ENR_CACHE[key] = df.copy()
#             return df
#         except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
#             # Transient errors from HTTP/JSON/file handling inside gseapy
#             if a == retries - 1:
#                 # Give up: return empty so pipeline continues
#                 return pd.DataFrame()
#             time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

#     return pd.DataFrame()

# # ---------------- 2) Minimal bootstrap to record robust terms ----------------
# def get_robust_terms(communities_HGNC, run_enrichment_func,
#                      R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
#     """
#     Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
#     Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
#     """
#     rng = np.random.default_rng(seed)
#     rows = []

#     for cid, community in enumerate(communities_HGNC):
#         n = len(community)
#         if n == 0:
#             continue
#         drop_k = max(1, int(np.floor(leaveout * n)))
#         counts = Counter()

#         for _ in range(R):
#             # Jackknife subset (ensure not empty)
#             keep = np.ones(n, dtype=bool)
#             keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
#             sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
#             if len(sub) == 0:
#                 continue

#             df = run_enrichment_safe(run_enrichment_func, sub)
#             if df is None or df.empty:
#                 continue

#             # Your function already returns significant terms; just count them.
#             # If it includes multiple libraries, preserve Gene_set to disambiguate names.
#             if 'Term' not in df.columns:
#                 continue  # be defensive

#             if 'Gene_set' in df.columns:
#                 terms = (df[['Term', 'Gene_set']]
#                          .dropna()
#                          .drop_duplicates()
#                          .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
#                          .tolist())
#             else:
#                 terms = df['Term'].dropna().drop_duplicates().tolist()

#             counts.update(terms)

#             # tiny pause helps with API rate limits if your func calls Enrichr internally
#             time.sleep(0.03)

#         # Keep only robust terms
#         for t, c in counts.items():
#             freq = c / max(R, 1)
#             if freq >= recurrence_cutoff:
#                 if '|' in t:
#                     term, gene_set = t.split('|', 1)
#                     rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
#                 else:
#                     rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

#     return (pd.DataFrame(rows)
#               .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
#               .reset_index(drop=True))

In [169]:
# twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
#                                 R=25, leaveout=0.1, recurrence_cutoff=0)

In [170]:
# twr3

In [171]:
# terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
#                                 R=10, leaveout=0.1, recurrence_cutoff=0)

In [172]:
# terms_with_recurrence

In [173]:
# # rename important terms to match terms_with_recurrence
# important_terms = important_terms.rename(columns={'index': 'community_id'})
# important_terms = important_terms.rename(columns={'Term': 'term'})

In [174]:
# terms_with_rec_merged = important_terms.merge(
#     terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
#     on=['community_id', 'term', 'Gene_set'],
#     how='left'
# )

# terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

# terms_with_rec_merged = terms_with_rec_merged.sort_values(
#     ['community_id', 'recurrence'],
#     ascending=[True, False]
# ).reset_index(drop=True)

In [175]:
# terms_with_rec_merged

In [176]:
# community_summary = (
#     terms_with_rec_merged
#     .groupby("community_id")["recurrence"]
#     .agg(mean_recurrence="mean", term_count="count")
#     .reset_index()
# )

# print(community_summary)

In [177]:
# display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [178]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [179]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))